<a href="https://colab.research.google.com/github/N-S-Wickramanayaka/Brahmee-transformer/blob/main/Random_forest_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the exact path to where your zip file is stored in Drive
# Replace 'dataset.zip' with the actual name of your zip file
zip_path = '/content/drive/MyDrive/rendered_stone.zip'

Mounted at /content/drive


In [3]:
import zipfile
import cv2
import numpy as np

images = []
labels = []

# Open the zip file in read mode
with zipfile.ZipFile(zip_path, 'r') as archive:
    # Get a list of all file paths inside the zip
    for file_path in archive.namelist():

        # Check if the path points to an image file (ignoring system files like __MACOSX)
        if (file_path.endswith('.png') or file_path.endswith('.jpg')) and '__MACOSX' not in file_path:

            # Extract the folder name (class label) from the path
            # Example path: 'dataset/ba/image1.png' -> folder name is 'ba'
            path_parts = file_path.split('/')
            if len(path_parts) >= 2:
                folder_name = path_parts[-2] # Second to last item is the folder name

                # Read the raw image bytes from the zip archive
                with archive.open(file_path) as file:
                    img_data = file.read()

                    # Convert raw bytes to a numpy array and decode it as a grayscale image
                    img_array = np.frombuffer(img_data, np.uint8)
                    img = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)

                    if img is not None:
                        # Resize to a uniform size (e.g., 64x64)
                        img_resized = cv2.resize(img, (64, 64))

                        images.append(img_resized)
                        labels.append(folder_name)

# Convert lists to numpy arrays
X = np.array(images)
y = np.array(labels)

print(f"Loaded {len(X)} images belonging to classes: {np.unique(y)}")

Loaded 3575 images belonging to classes: ['ba' 'dha' 'ha' 'la' 'ma' 'na' 'pa' 'ra' 'sa' 'sha' 'tha' 'wa' 'ya']


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Flatten
num_samples = X.shape[0]
X_flattened = X.reshape(num_samples, -1) / 255.0

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(
    X_flattened, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Train
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# 4. Evaluate
y_pred = rf_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

          ba       0.34      0.36      0.35        55
         dha       0.23      0.24      0.23        55
          ha       0.10      0.09      0.10        55
          la       0.20      0.20      0.20        55
          ma       0.44      0.42      0.43        55
          na       0.27      0.35      0.30        55
          pa       0.15      0.13      0.14        55
          ra       0.31      0.47      0.38        55
          sa       0.29      0.25      0.27        55
         sha       0.07      0.05      0.06        55
         tha       0.14      0.15      0.14        55
          wa       0.41      0.35      0.38        55
          ya       0.30      0.24      0.26        55

    accuracy                           0.25       715
   macro avg       0.25      0.25      0.25       715
weighted avg       0.25      0.25      0.25       715



**Enhancement to the basic model of Random forest with HOG feature**

In [2]:
from google.colab import drive
import os
import zipfile
import cv2
import numpy as np
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# STEP 1: Mount Google Drive & Define Path
# ==========================================
drive.mount('/content/drive')
zip_path = '/content/drive/MyDrive/rendered_stone.zip' # Update this to your file path

# Arrays to hold our extracted features and corresponding labels
X_features = []
y_labels = []

# ==========================================
# STEP 2: Load Zip & Extract HOG Features
# ==========================================
print("Extracting HOG features from images... This may take a moment.")

with zipfile.ZipFile(zip_path, 'r') as archive:
    for file_path in archive.namelist():

        # Filter for valid image formats and exclude system files
        if (file_path.endswith('.png') or file_path.endswith('.jpg')) and '__MACOSX' not in file_path:

            path_parts = file_path.split('/')
            if len(path_parts) >= 2:
                folder_name = path_parts[-2] # Folder name acts as the label

                with archive.open(file_path) as file:
                    img_data = file.read()
                    img_array = np.frombuffer(img_data, np.uint8)
                    img = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)

                    if img is not None:
                        # 1. Resize to a consistent size (64x64 works well for HOG)
                        img_resized = cv2.resize(img, (64, 64))

                        # 2. Extract HOG features
                        # These parameters break the image into 8x8 pixel cells and calculate edge directions
                        hog_vector = hog(
                            img_resized,
                            orientations=9,
                            pixels_per_cell=(8, 8),
                            cells_per_block=(2, 2),
                            visualize=False
                        )

                        # hog_vector is already a flattened 1D array of shape (1764,)
                        X_features.append(hog_vector)
                        y_labels.append(folder_name)

# Convert to numpy arrays
X = np.array(X_features)
y = np.array(y_labels)

print(f"Extraction complete! Feature matrix shape: {X.shape}")
print(f"Total samples: {X.shape[0]}, Features per image: {X.shape[1]}")

# ==========================================
# STEP 3: Split Dataset
# ==========================================
# Since HOG handles the feature processing, we don't need to manually flatten or divide by 255
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ==========================================
# STEP 4: Train Random Forest Classifier
# ==========================================
print("Training Random Forest on HOG features...")
rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# ==========================================
# STEP 5: Evaluate Model
# ==========================================
y_pred = rf_model.predict(X_test)

print("\n=== NEW CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

Mounted at /content/drive
Extracting HOG features from images... This may take a moment.
Extraction complete! Feature matrix shape: (3575, 1764)
Total samples: 3575, Features per image: 1764
Training Random Forest on HOG features...

=== NEW CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

          ba       0.89      0.62      0.73        55
         dha       0.50      0.65      0.57        55
          ha       0.74      0.71      0.72        55
          la       0.57      0.58      0.58        55
          ma       0.65      0.71      0.68        55
          na       0.67      0.62      0.64        55
          pa       0.56      0.60      0.58        55
          ra       0.59      0.60      0.59        55
          sa       0.75      0.69      0.72        55
         sha       0.58      0.64      0.61        55
         tha       0.68      0.76      0.72        55
          wa       0.62      0.53      0.57        55
          ya       0.64      